# Continue the latest published Beacon adapter

Resolves the current Hugging Face main revision at run start, verifies its weight hash, and continues those weights. Use Save Version > Save & Run All on Kaggle with GPU and Internet enabled. Update the attached training dataset for your next training round before running.


In [ ]:
import os, sys, json, glob, zipfile, subprocess, hashlib, time
from pathlib import Path
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
STARTED = time.time()
WORK = Path('/kaggle/working/beacon_v3_continue')
WORK.mkdir(exist_ok=True)
archives = glob.glob('/kaggle/input/**/beacon_training_v3.zip', recursive=True)
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        for member in z.infolist():
            target = (WORK / member.filename).resolve()
            if not target.is_relative_to(WORK.resolve()): raise ValueError('Unsafe archive path')
        z.extractall(WORK)
elif len(archives) == 0:
    # Kaggle sometimes exposes the uploaded ZIP as extracted files.
    found = list(Path('/kaggle/input').rglob('bundle_manifest.json'))
    if len(found) != 1: raise RuntimeError('Attach exactly one beacon_training_v3 dataset first.')
    import shutil
    source = found[0].parent
    for rel in json.loads(found[0].read_text())['files']:
        dest=WORK/rel; dest.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(source/rel,dest)
    shutil.copy2(found[0],WORK/'bundle_manifest.json')
else:
    raise RuntimeError('Multiple training bundles attached; keep only the intended version.')
os.chdir(WORK)
for rel, expected in json.loads(Path('bundle_manifest.json').read_text())['files'].items():
    assert hashlib.sha256(Path(rel).read_bytes()).hexdigest() == expected, 'Hash mismatch: '+rel
print('Training bundle verified.')
TRAINER_SOURCE='"""QLoRA fine-tune of Qwen2.5-1.5B-Instruct on the Beacon qualification data (shared by the Colab notebook and local runs).\n\nUsage: python slm/train.py --out slm/runs/<name> [--epochs 2] [--max-len 2048]\nLoss is computed on the JSON answer only (prompt tokens masked). Writes the adapter and run_metadata.json\n(peak VRAM, wall-clock, GPU, library versions, data hashes, training metrics).\n"""\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport random\nimport sys\nimport time\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT))\nfrom slm.prompting import messages, target_text, facts_messages, facts_target_text  # noqa: E402\n\nBASE = \'Qwen/Qwen2.5-1.5B-Instruct\'\nSEED = 20260928\n\ndef load(path):\n    return [json.loads(l) for l in Path(path).read_text(\'utf-8\').splitlines() if l.strip()]\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--out\', required=True); parser.add_argument(\'--epochs\', type=float, default=2)\n    parser.add_argument(\'--max-len\', type=int, default=3072); parser.add_argument(\'--lr\', type=float, default=5e-5)\n    parser.add_argument(\'--grad-accum\', type=int, default=16)\n    parser.add_argument(\'--format\', choices=[\'full\', \'facts\'], default=\'full\', help=\'facts: the model extracts, the rules score\')\n    parser.add_argument(\'--lang\', default=None, help=\'train/evaluate only on this language code, e.g. en\')\n    parser.add_argument(\'--data-dir\', type=Path, default=ROOT / \'slm\' / \'data\' / \'v3\', help=\'Directory containing train.jsonl and dev.jsonl\')\n    parser.add_argument(\'--check-only\', action=\'store_true\', help=\'Validate data and token lengths without loading or training model weights\')\n    parser.add_argument(\'--init-adapter\', type=Path, help=\'Continue training existing adapter weights; optimizer/scheduler start fresh\')\n    parser.add_argument(\'--fresh-adapter\', action=\'store_true\', help=\'Explicit opt-in to initialize new adapter weights instead of continuing the latest published adapter\')\n    args = parser.parse_args()\n    if args.fresh_adapter and args.init_adapter:\n        parser.error(\'--fresh-adapter and --init-adapter are mutually exclusive\')\n    parent_revision = None\n    if not args.init_adapter and not args.fresh_adapter:\n        from huggingface_hub import HfApi, snapshot_download\n        repo = \'abidansari5594/beacon-qualification-qwen2.5-1.5b-qlora\'\n        info = HfApi().model_info(repo, files_metadata=True)\n        args.init_adapter = Path(snapshot_download(repo, revision=info.sha, allow_patterns=[\n            \'adapter*\', \'tokenizer*\', \'vocab.json\', \'merges.txt\', \'special_tokens_map.json\', \'added_tokens.json\', \'chat_template.jinja\']))\n        expected = next(s.lfs.sha256 for s in info.siblings if s.rfilename == \'adapter_model.safetensors\')\n        if hashlib.sha256((args.init_adapter/\'adapter_model.safetensors\').read_bytes()).hexdigest() != expected:\n            raise ValueError(\'Published adapter hash mismatch\')\n        parent_revision = {\'repo\': repo, \'revision\': info.sha}\n        print(\'Latest published parent:\', info.sha, flush=True)\n    started = time.time()\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments\n    from peft import LoraConfig, PeftModel, get_peft_model, get_peft_model_state_dict, prepare_model_for_kbit_training\n    import transformers, peft, bitsandbytes\n    random.seed(SEED); torch.manual_seed(SEED)\n    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)\n    parent = None\n    if args.init_adapter:\n        config = json.loads((args.init_adapter / \'adapter_config.json\').read_text(\'utf-8\'))\n        if config[\'base_model_name_or_path\'] != BASE or config[\'peft_type\'] != \'LORA\':\n            raise ValueError(\'Parent adapter must be a LoRA adapter for \' + BASE)\n        if args.init_adapter.resolve() == (out / \'adapter\').resolve():\n            raise ValueError(\'Output must not overwrite the parent adapter\')\n        weights = args.init_adapter / \'adapter_model.safetensors\'\n        if not weights.is_file(): raise ValueError(\'Missing parent adapter_model.safetensors\')\n        parent = {\'path\':str(args.init_adapter), \'weights_sha256\':hashlib.sha256(weights.read_bytes()).hexdigest(),\n                  \'config_sha256\':hashlib.sha256((args.init_adapter / \'adapter_config.json\').read_bytes()).hexdigest(),\n                  \'optimizer_state_restored\':False}\n        print(\'CONTINUATION: loading existing adapter weights; optimizer/scheduler start fresh. Parent SHA256: \' + parent[\'weights_sha256\'], flush=True)\n    tokenizer_kwargs = {}\n    if args.init_adapter and int(transformers.__version__.split(\'.\')[0]) < 5:\n        tokenizer_config = json.loads((args.init_adapter / \'tokenizer_config.json\').read_text(\'utf-8\'))\n        extra = tokenizer_config.get(\'extra_special_tokens\')\n        if isinstance(extra, list):\n            # Transformers 5 saves this as a list; Transformers 4 expects a mapping.\n            # Keep the same special tokens and the original tokenizer vocabulary.\n            tokenizer_kwargs = {\'extra_special_tokens\': {}, \'additional_special_tokens\': extra}\n    tok = AutoTokenizer.from_pretrained(str(args.init_adapter) if args.init_adapter else BASE, **tokenizer_kwargs)\n\n    def encode(row):\n        msgs = facts_messages(row[\'transcript\']) if args.format == \'facts\' else messages(row[\'transcript\'])\n        prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)\n        prompt = tok(prompt, add_special_tokens=False)[\'input_ids\']\n        answer = tok((facts_target_text if args.format == \'facts\' else target_text)(row[\'target\']) + tok.eos_token, add_special_tokens=False)[\'input_ids\']\n        if len(prompt) + len(answer) > args.max_len:\n            raise ValueError(f"{row[\'id\']}: {len(prompt)+len(answer)} tokens exceeds --max-len {args.max_len}; increase max-len, do not truncate the JSON target")\n        return {\'input_ids\': prompt + answer, \'labels\': [-100] * len(prompt) + answer}\n\n    keep = lambda rows: [r for r in rows if args.lang is None or r.get(\'language\') == args.lang]\n    train_rows = keep(load(args.data_dir / \'train.jsonl\'))\n    dev_rows = keep(load(args.data_dir / \'dev.jsonl\'))\n    if not train_rows or not dev_rows:\n        raise ValueError(\'Training and development sets must both contain examples after language filtering\')\n    if {r[\'id\'] for r in train_rows} & {r[\'id\'] for r in dev_rows} or {r[\'family\'] for r in train_rows} & {r[\'family\'] for r in dev_rows}:\n        raise ValueError(\'Train/dev overlap in IDs or scenario families\')\n    train = [encode(r) for r in train_rows]\n    dev = [encode(r) for r in dev_rows]\n    lengths = sorted(len(x[\'input_ids\']) for x in train)\n    truncated = sum(len(x[\'input_ids\']) >= args.max_len for x in train)\n    print(f\'train {len(train)} dev {len(dev)} tokens p50 {lengths[len(lengths)//2]} p95 {lengths[int(.95*len(lengths))]} max {lengths[-1]} truncated {truncated}\', flush=True)\n    if args.check_only:\n        print(\'Preflight passed; no training performed.\', flush=True)\n        return\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA GPU required for this QLoRA run. Enable a GPU in Kaggle settings.\')\n    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id\n\n    def collate(batch):\n        n = max(len(x[\'input_ids\']) for x in batch)\n        return {\'input_ids\': torch.tensor([x[\'input_ids\'] + [pad] * (n - len(x[\'input_ids\'])) for x in batch]),\n                \'attention_mask\': torch.tensor([[1] * len(x[\'input_ids\']) + [0] * (n - len(x[\'input_ids\'])) for x in batch]),\n                \'labels\': torch.tensor([x[\'labels\'] + [-100] * (n - len(x[\'labels\'])) for x in batch])}\n\n    torch.cuda.reset_peak_memory_stats()\n    model = AutoModelForCausalLM.from_pretrained(BASE, device_map={\'\': 0}, torch_dtype=torch.float16, quantization_config=BitsAndBytesConfig(\n        load_in_4bit=True, bnb_4bit_quant_type=\'nf4\', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True))\n    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)\n    model.config.use_cache = False\n    if args.init_adapter:\n        model = PeftModel.from_pretrained(model, str(args.init_adapter), is_trainable=True)\n        # Fail closed if the saved weights were not actually restored before training.\n        from safetensors.torch import load_file\n        saved = load_file(str(args.init_adapter / \'adapter_model.safetensors\'), device=\'cpu\')\n        loaded = get_peft_model_state_dict(model)\n        if set(saved) != set(loaded): raise ValueError(\'Parent adapter tensor keys differ after load\')\n        for key, value in saved.items():\n            if not torch.equal(value, loaded[key].detach().cpu().to(value.dtype)):\n                raise ValueError(\'Parent adapter weights did not restore exactly: \' + key)\n        if not any(p.requires_grad for p in model.parameters()): raise ValueError(\'Adapter is frozen\')\n        print(f\'CONTINUATION VERIFIED: {len(saved)} saved adapter tensors restored exactly; adapter is trainable.\', flush=True)\n        del saved, loaded\n    else:\n        model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type=\'CAUSAL_LM\',\n            target_modules=[\'q_proj\', \'k_proj\', \'v_proj\', \'o_proj\', \'gate_proj\', \'up_proj\', \'down_proj\']))\n    model.print_trainable_parameters()\n    training = TrainingArguments(output_dir=str(out / \'checkpoints\'), per_device_train_batch_size=1, per_device_eval_batch_size=1,\n        gradient_accumulation_steps=args.grad_accum, num_train_epochs=args.epochs, learning_rate=args.lr, warmup_steps=max(1, round(0.05 * args.epochs * len(train) / args.grad_accum)),\n        lr_scheduler_type=\'cosine\', fp16=True, logging_steps=5, eval_strategy=\'epoch\', save_strategy=\'epoch\', save_total_limit=1,\n        seed=SEED, report_to=[], dataloader_pin_memory=False)\n    trainer = Trainer(model=model, args=training, train_dataset=train, eval_dataset=dev, data_collator=collate)\n    train_started = time.time(); result = trainer.train(); train_seconds = time.time() - train_started\n    model.save_pretrained(out / \'adapter\'); tok.save_pretrained(out / \'adapter\')\n    data = {n: hashlib.sha256((args.data_dir / f\'{n}.jsonl\').read_bytes()).hexdigest() for n in [\'train\', \'dev\']}\n    meta = {\'base_model\': BASE, \'licence\': \'Apache-2.0\', \'method\': \'QLoRA NF4 r16 a32, batch 1 x grad-accum \' + str(args.grad_accum),\n            \'format\': args.format, \'lang\': args.lang, \'data_dir\': str(args.data_dir), \'epochs\': args.epochs, \'max_len\': args.max_len, \'seed\': SEED, \'train_examples\': len(train), \'dev_examples\': len(dev),\n            \'training_mode\':\'continue_adapter\' if parent else \'fresh_adapter\', \'parent_adapter\':parent, \'parent_repository\':parent_revision,\n            \'train_seconds\': round(train_seconds), \'total_seconds\': round(time.time() - started),\n            \'peak_vram_gib\': round(torch.cuda.max_memory_allocated() / 2**30, 2), \'peak_reserved_gib\': round(torch.cuda.max_memory_reserved() / 2**30, 2),\n            \'gpu\': torch.cuda.get_device_name(0), \'platform\': platform.platform(),\n            \'versions\': {\'python\': platform.python_version(), \'torch\': torch.__version__, \'transformers\': transformers.__version__,\n                         \'peft\': peft.__version__, \'bitsandbytes\': bitsandbytes.__version__},\n            \'data_sha256\': data, \'train_metrics\': result.metrics, \'log_history\': trainer.state.log_history}\n    (out / \'run_metadata.json\').write_text(json.dumps(meta, indent=1), \'utf-8\')\n    print(json.dumps({k: v for k, v in meta.items() if k != \'log_history\'}, indent=1))\n\nif __name__ == \'__main__\':\n    main()\n'
Path('slm/train.py').write_text(TRAINER_SOURCE,encoding='utf-8')
assert hashlib.sha256(Path('slm/train.py').read_bytes()).hexdigest()=='bfafbdbc1eb5c81ee01b8abbc5ca52c7d5d376e9c5002a55b19b596cbbaa9881'
bundle=json.loads(Path('bundle_manifest.json').read_text())
bundle['files']['slm/train.py']=hashlib.sha256(Path('slm/train.py').read_bytes()).hexdigest()
Path('bundle_manifest.json').write_text(json.dumps(bundle,indent=2))

def run(command):
    proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env={**os.environ,'PYTHONUNBUFFERED':'1'})
    for line in proc.stdout: print(line,end='',flush=True)
    if proc.wait(): raise RuntimeError('Command failed: '+' '.join(command))
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','peft==0.17.1',
     'accelerate==1.10.1','bitsandbytes==0.48.1','pydantic>=2.9,<3'])
from huggingface_hub import snapshot_download, HfApi
PARENT_REPO='abidansari5594/beacon-qualification-qwen2.5-1.5b-qlora'
parent_info=HfApi().model_info(PARENT_REPO,files_metadata=True)
PARENT_REVISION=parent_info.sha
PARENT_SHA=next(s.lfs.sha256 for s in parent_info.siblings if s.rfilename=='adapter_model.safetensors')
snapshot_download(repo_id=PARENT_REPO, revision=PARENT_REVISION, local_dir='parent_adapter',
                  allow_patterns=['adapter_config.json','adapter_model.safetensors','chat_template.jinja','tokenizer_config.json','tokenizer.json'])
assert hashlib.sha256(Path('parent_adapter/adapter_model.safetensors').read_bytes()).hexdigest()==PARENT_SHA, 'Wrong adapter weights'
print('VERIFIED HUGGING FACE PARENT:',PARENT_REPO,'revision',PARENT_REVISION,'SHA256',PARENT_SHA)
bundle=json.loads(Path('bundle_manifest.json').read_text())
bundle['parent_repo']=PARENT_REPO; bundle['parent_revision']=PARENT_REVISION; bundle['parent_weights_sha256']=PARENT_SHA
Path('bundle_manifest.json').write_text(json.dumps(bundle,indent=2))

import torch
assert torch.cuda.is_available(), 'Enable a GPU in Kaggle Settings before running.'
print('GPU:',torch.cuda.get_device_name(0))
print('GPU memory GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,1))


In [ ]:
# Validate counts, split separation, hashes and complete JSON target lengths before training.
manifest=json.loads(Path('slm/data/v3/manifest.json').read_text())
splits={}
for split in ['train','dev']:
    p=Path(f'slm/data/v3/{split}.jsonl')
    assert hashlib.sha256(p.read_bytes()).hexdigest()==manifest['splits'][split]['sha256']
    splits[split]=[json.loads(line) for line in p.read_text().splitlines() if line.strip()]
    english=[r for r in splits[split] if r['language']=='en']
    print(split,'all:',len(splits[split]),'English subset:',len(english))
assert not {r['family'] for r in splits['train']} & {r['family'] for r in splits['dev']}
assert len([r for r in splits['train'] if r['language']=='en'])>2000
ARGS=['--out','slm/runs/kaggle-v3-continued','--data-dir','slm/data/v3','--format','full',
      '--init-adapter','parent_adapter','--lr','5e-5','--max-len','3072','--epochs','2','--grad-accum','16']
run([sys.executable,'slm/train.py',*ARGS,'--check-only'])


In [ ]:
# Train only. Progress and validation loss are printed below.
run([sys.executable,'slm/train.py',*ARGS])


In [ ]:
# Save adapter immediately. No held-out test or report is run.
run_dir=Path('slm/runs/kaggle-v3-continued')
meta=json.loads((run_dir/'run_metadata.json').read_text())
assert meta['train_examples']==len(splits['train']) and meta['dev_examples']==len(splits['dev'])
assert meta['training_mode']=='continue_adapter'
assert meta['parent_adapter']['weights_sha256']==PARENT_SHA
meta['parent_repo']=PARENT_REPO; meta['parent_revision']=PARENT_REVISION
meta['notebook_seconds']=round(time.time()-STARTED)
meta['test_status']='not run in this training notebook; evaluate the new release separately'
(run_dir/'run_metadata.json').write_text(json.dumps(meta,indent=2))
assert (run_dir/'adapter/adapter_config.json').exists()
assert any((run_dir/'adapter').glob('*.safetensors'))
output=Path('/kaggle/working/beacon_v3_continued_adapter.zip')
with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted((run_dir/'adapter').rglob('*')):
        if p.is_file(): z.write(p, str(p.relative_to(run_dir)))
    z.write(run_dir/'run_metadata.json','run_metadata.json')
    z.write('slm/data/v3/manifest.json','data_manifest.json')
    z.write('bundle_manifest.json','bundle_manifest.json')
with zipfile.ZipFile(output) as z: assert z.testzip() is None
print('TRAINING COMPLETE. Download:',output)
print('Examples:',meta['train_examples'],'GPU:',meta['gpu'],'training seconds:',meta['train_seconds'])
